# QuantiPhy validation EDA

This notebook reproduces the parser, video grouping, and exploratory analysis included in the project.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
from quantiphy_baseline.dataio import load_dataset
from quantiphy_baseline.parsers.question_parser import parse_question_row
from quantiphy_baseline.grouping import group_questions_by_video
from quantiphy_baseline.eda import parsed_questions_to_frame, video_groups_to_frame, build_eda


## 1. Load and normalize the official validation CSV


In [ ]:
raw = load_dataset(PROJECT_ROOT / 'data/raw/validation_dataset.csv')
print(raw.shape)
raw.head()


## 2. Parse questions and group by video


In [ ]:
parsed = [parse_question_row(row) for row in raw.to_dict('records')]
groups = group_questions_by_video(parsed)
qa = parsed_questions_to_frame([q for group in groups for q in group.questions])
videos = video_groups_to_frame(groups)
print(f'{len(qa)} questions across {len(videos)} videos')
qa.head()


## 3. Dataset balance


In [ ]:
display(qa['video_source'].value_counts(dropna=False).to_frame('questions'))
display(qa['inference_type'].value_counts(dropna=False).to_frame('questions'))
display(qa['quantity_family'].value_counts(dropna=False).to_frame('questions'))
display(pd.crosstab(qa['inference_type'], qa['quantity_family'], margins=True))


## 4. Questions per video and shared context


In [ ]:
display(videos.sort_values('num_questions', ascending=False))
qa.loc[qa['output_unit_source'] == 'video_context', [
    'qa_id', 'video_id', 'question', 'quantity_subtype', 'output_unit', 'warnings'
]]


## 5. Parser quality checks


In [ ]:
quality = pd.Series({
    'unknown_quantity': (qa['quantity_family'] == 'unknown').sum(),
    'missing_unit': qa['output_unit'].isna().sum(),
    'no_entity': (qa['target_entity_count'] == 0).sum(),
    'low_confidence': (qa['parse_confidence'] < 0.75).sum(),
    'mean_confidence': qa['parse_confidence'].mean(),
})
quality


## 6. Numerical scale


In [ ]:
positive = qa.loc[qa['ground_truth'] > 0, 'ground_truth']
pd.Series({
    'min': positive.min(),
    'median': positive.median(),
    'max': positive.max(),
    'observed_decades': positive.map(lambda x: __import__('math').log10(x)).max() - positive.map(lambda x: __import__('math').log10(x)).min(),
})


## 7. Generate the complete EDA report


In [ ]:
summary = build_eda(
    [q for group in groups for q in group.questions],
    groups,
    PROJECT_ROOT / 'outputs/eda',
)
summary
